# LeetCode 3114 — Latest Time You Can Obtain After Replacing Characters bằng Python

Link đề: https://leetcode.com/problems/latest-time-you-can-obtain-after-replacing-characters/ · Độ khó: **Easy** · Chủ đề: String, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def findLatestTime(self, s: str) -> str:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi `s` dạng `"HH:MM"` (giờ định dạng **12 giờ**, từ `00:00` đến `11:59`) có một số ký tự là dấu `?`; hãy **thay các `?` bằng chữ số** để thu được **thời gian hợp lệ muộn nhất** có thể.
2. Nắm các công cụ Python: duyệt chuỗi theo **chỉ số**, so khớp **mẫu** có ký tự đại diện `?`, định dạng số có số `0` đứng đầu (`f"{h:02d}"`), `itertools.product` để thử mọi cách điền, và `max` trên chuỗi.
3. Đi từ cách **duyệt mọi thời gian từ muộn tới sớm, trả về thời gian đầu tiên khớp mẫu** → **giờ và phút tách rời nhau: tìm riêng** → **thử mọi cách điền `?` bằng `product` rồi chọn lớn nhất hợp lệ** → **tham lam từng chữ số theo luật** (`H1`, `H2`, `M1`, `M2`).
4. Hiểu mấu chốt: giờ và phút **độc lập**; chữ số lớn hơn ở vị trí đầu cho thời gian muộn hơn (so sánh theo từ điển trong chuỗi độ dài cố định), nhưng **chữ số giờ phải giữ sao cho giờ ≤ 11**, nên các ca `?` ở giờ cần **luật phụ thuộc lẫn nhau**.
5. Tránh các lỗi kinh điển: tưởng giờ tối đa `23` (24 giờ), điền `H1 = '1'` khi `H2 >= '2'`, điền `H2 = '9'` khi `H1 = '1'`, quên rằng phút tối đa `59` (`M1 = '5'`), và sai thứ tự xử lý khi `H1` và `H2` đều là `?`.

## 2. Đề bài & ví dụ

Cho một chuỗi `s` có dạng `"HH:MM"` biểu diễn **thời gian theo định dạng 12 giờ**, trong đó `HH` nằm giữa `00` và `11` và `MM` nằm giữa `00` và `59`. Một số chữ số trong `s` bị thay bằng dấu **`?`**.

Hãy **thay mọi dấu `?`** bằng chữ số sao cho thu được **thời gian hợp lệ muộn nhất** có thể. Trả về chuỗi kết quả.

Đề đảm bảo có ít nhất một thời gian hợp lệ có thể tạo được.

**Ví dụ 1**

- Input: `s = "1?:?4"`
- Output: `"11:54"`
- Giải thích: giờ có dạng `1?`: lớn nhất hợp lệ là `11` (vì `12`, `13`, ... không hợp lệ ở định dạng 12 giờ). Phút có dạng `?4`: lớn nhất là `54`. Kết quả `"11:54"`.

**Ví dụ 2**

- Input: `s = "0?:5?"`
- Output: `"09:59"`
- Giải thích: giờ `0?` lớn nhất là `09`; phút `5?` lớn nhất là `59`.

**Ví dụ 3 (tự dựng)**

- Input: `s = "??:??"`
- Output: `"11:59"`
- Giải thích: thời gian muộn nhất của định dạng 12 giờ là `11:59`.

**Ràng buộc:** `s.length == 5`; `s[2] == ':'`; mọi ký tự khác `:` là chữ số hoặc `'?'`; dữ liệu được tạo sao cho **có ít nhất một thời gian hợp lệ** từ `00:00` đến `11:59` khớp với `s`.

In [1]:
import random
import re
from itertools import product

examples = [
    ("1?:?4", "11:54"),
    ("0?:5?", "09:59"),
    ("??:??", "11:59"),
]
for s, expected in examples:
    print(repr(s), "-> kỳ vọng", repr(expected))

'1?:?4' -> kỳ vọng '11:54'
'0?:5?' -> kỳ vọng '09:59'
'??:??' -> kỳ vọng '11:59'


## 3. Kiến thức nền

### 3.1. Định dạng 12 giờ và độ rộng cố định

Trong bài này giờ chạy từ `00` đến `11` (12 giá trị) và phút từ `00` đến `59` (60 giá trị), tổng `12 × 60 = 720` thời gian. Chuỗi luôn có **độ dài cố định 5** (`HH:MM`), nên hai thời gian bất kỳ so sánh **như chuỗi (từ điển)** cho kết quả giống so sánh thời gian thật: `"09:59" < "10:00"`.

In [2]:
print("09:59" < "10:00", "11:54" < "11:59")      # True True
all_times = [f"{h:02d}:{m:02d}" for h in range(12) for m in range(60)]
print(len(all_times), all_times[0], all_times[-1])     # 720 00:00 11:59
print(all_times == sorted(all_times))                  # True: chuỗi theo thứ tự thời gian

True True
720 00:00 11:59
True


### 3.2. Định dạng số có `0` đứng đầu: `f"{n:02d}"`

`f"{n:02d}"` in số nguyên `n` thành **ít nhất hai chữ số**, thêm `0` ở đầu nếu cần (`5` thành `"05"`). Cách khác: `str(n).zfill(2)`.

In [3]:
print(f"{5:02d}", f"{11:02d}", f"{0:02d}")        # 05 11 00
print(str(5).zfill(2), str(59).zfill(2))          # 05 59

05 11 00
05 59


### 3.3. Khớp mẫu có ký tự đại diện `?`

Ký tự `'?'` trong `s` khớp **mọi chữ số**. Một thời gian `t` "khớp" mẫu `s` nếu với **mọi vị trí** `i`, hoặc `s[i] == '?'`, hoặc `s[i] == t[i]`. Ta viết hàm `matches(s, t)` bằng `all(...)` và `zip`.

In [4]:
def matches(pattern: str, time: str) -> bool:
    return all(p == "?" or p == t for p, t in zip(pattern, time))


print(matches("1?:?4", "11:54"))     # True
print(matches("1?:?4", "12:54"))     # True (về mẫu), nhưng 12:54 không hợp lệ ở định dạng 12 giờ
print(matches("1?:?4", "10:55"))     # False (phút kết thúc bằng 5, mẫu cần 4)

True
True
False


### 3.4. Điền `?` bằng mọi tổ hợp chữ số: `itertools.product`

`product("0123456789", repeat=k)` sinh **mọi tổ hợp** `k` chữ số (`10^k` tổ hợp). Nếu `s` có `k` dấu `?`, ta thử `10^k` cách điền (tối đa `10^4 = 10000` vì còn ít nhất dấu `:`).

In [5]:
print(len(list(product("0123456789", repeat=2))))        # 100
print(list(product("01", repeat=2)))                      # [('0','0'), ('0','1'), ('1','0'), ('1','1')]

100
[('0', '0'), ('0', '1'), ('1', '0'), ('1', '1')]


### 3.5. `max` trên các chuỗi và biểu thức điều kiện

`max(["09:59", "10:00", "08:30"])` so theo từ điển và cho `"10:00"`. Biểu thức điều kiện `A if cond else B` sẽ dùng nhiều ở phần tham lam. Hàm `in` kiểm tra `ký_tự in "01?"` là một cách ngắn để hỏi "ký tự thuộc tập này".

In [6]:
print(max(["09:59", "10:00", "08:30"]))                # 10:00
for c in "0?5":
    print(c, c in "01?")                                # 0 True / ? True / 5 False

10:00
0 True
? True
5 False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Muốn thời gian **muộn nhất**, ta muốn **chữ số quan trọng nhất (bên trái)** lớn nhất có thể, rồi tới chữ số kế tiếp... nhưng phải giữ **thời gian hợp lệ**.

Xét `s = "1?:?4"`:

- Giờ `1?`: các giờ khớp và hợp lệ: `10`, `11` (không có `12` vì giờ ≤ 11) ⇒ lớn nhất `11`.
- Phút `?4`: các phút khớp: `04, 14, 24, 34, 44, 54` ⇒ lớn nhất `54`.

Kết quả `11:54`.

Xét `s = "0?:5?"`: giờ `0?`: `00..09` ⇒ `09`; phút `5?`: `50..59` ⇒ `59`; kết quả `09:59`.

Điểm quan trọng: **giờ và phút độc lập** (không ràng buộc lẫn nhau), nên ta có thể tìm giờ tốt nhất và phút tốt nhất **riêng**.

In [7]:
hours = [f"{h:02d}" for h in range(12)]
minutes = [f"{m:02d}" for m in range(60)]
hour_pat, min_pat = "1?", "?4"
cands_h = [h for h in hours if matches(hour_pat, h)]
cands_m = [m for m in minutes if matches(min_pat, m)]
print("giờ khớp:", cands_h, "-> lớn nhất", max(cands_h))
print("phút khớp:", cands_m, "-> lớn nhất", max(cands_m))

giờ khớp: ['10', '11'] -> lớn nhất 11
phút khớp: ['04', '14', '24', '34', '44', '54'] -> lớn nhất 54


## 5. Cách 1 — Brute Force / baseline: thử mọi thời gian từ muộn tới sớm

**Ý tưởng.** Có `720` thời gian hợp lệ. Ta duyệt từ **`11:59` giảm dần** tới `00:00`; thời gian **đầu tiên** khớp mẫu `s` chính là thời gian muộn nhất. Đúng theo định nghĩa, và `720 × 5` phép so sánh là rất nhỏ.

In [8]:
def find_latest_bruteforce(s: str) -> str:
    for h in range(11, -1, -1):                 # giờ từ 11 xuống 0
        for m in range(59, -1, -1):             # phút từ 59 xuống 0
            time = f"{h:02d}:{m:02d}"
            if matches(s, time):                # khớp mẫu: đây là thời gian muộn nhất
                return time
    return ""                                   # theo đề không bao giờ xảy ra


for s, expected in examples:
    print(repr(s), find_latest_bruteforce(s), "(kỳ vọng", expected, ")")

'1?:?4' 11:54 (kỳ vọng 11:54 )
'0?:5?' 09:59 (kỳ vọng 09:59 )
'??:??' 11:59 (kỳ vọng 11:59 )


**Độ phức tạp (Cách 1).** Tối đa `720` thời gian, mỗi lần so `5` ký tự ⇒ **Time `O(1)`** (hằng số `~3600`), **Space `O(1)`**.

**Nhận xét:** vì miền đầu vào có hằng số nhỏ nên brute force này **đã đủ để nộp**; nó cũng là "chuẩn" để kiểm tra các cách khác. Tuy nhiên ta nên học cả cách dùng **luật từng chữ số** (nền tảng cho bài khó hơn).

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — tách giờ và phút.** `HH` và `MM` độc lập: tìm `HH` muộn nhất khớp `s[0:2]` (trong `00..11`) và `MM` muộn nhất khớp `s[3:5]` (trong `00..59`).

**Bước 2 — luật từng chữ số (tham lam).** Chữ số bên trái có trọng số lớn hơn nên ưu tiên lớn nhất có thể mà **vẫn còn đáp án**:

- **`M2` (chữ số phút hàng đơn vị)**: `'?'` ⇒ `'9'`.
- **`M1` (chữ số phút hàng chục)**: `'?'` ⇒ `'5'` (phút tối đa `59`).
- **`H2` (chữ số giờ hàng đơn vị)**: `'?'` ⇒ nếu `H1 == '1'` thì `'1'` (giờ ≤ `11`), ngược lại (`H1 == '0'`) thì `'9'`.
- **`H1` (chữ số giờ hàng chục)**: `'?'` ⇒ nếu `H2` là `'?'` hoặc `H2 <= '1'` thì `'1'` (cho `10`, `11`), ngược lại (`H2 >= '2'`) thì `'0'` (cho `02..09`).

Chú ý `H1` phụ thuộc `H2` **ban đầu** (chưa điền), còn `H2` phụ thuộc `H1` **đã điền**. Vì vậy xử lý `H1` trước, dùng giá trị gốc của `H2`; rồi xử lý `H2`, dùng `H1` đã quyết.

In [9]:
# Kiểm tra luật H1 / H2 trên mọi (H1, H2) có thể: so với thử mọi giờ hợp lệ
hours = [f"{h:02d}" for h in range(12)]


def best_hour_by_rule(h1, h2):
    a, b = h1, h2
    if a == "?":
        a = "1" if (b == "?" or b <= "1") else "0"
    if b == "?":
        b = "1" if a == "1" else "9"
    return a + b


def best_hour_brute(h1, h2):
    cands = [h for h in hours if (h1 in ("?", h[0])) and (h2 in ("?", h[1]))]
    return max(cands) if cands else None


for h1 in "?01":
    for h2 in "?0123456789":
        expected = best_hour_brute(h1, h2)
        if expected is not None:                       # chỉ xét cặp có ít nhất một giờ hợp lệ
            assert best_hour_by_rule(h1, h2) == expected, (h1, h2)
print("luật H1/H2 khớp brute force trên mọi mẫu giờ hợp lệ")

luật H1/H2 khớp brute force trên mọi mẫu giờ hợp lệ


## 7. Các cách tiếp theo

### Cách 2 — Tìm riêng giờ và phút

**Giải thích từng đoạn.**

1. Danh sách giờ `hours = ["11", "10", ..., "00"]` (giảm dần) và phút `minutes = ["59", ..., "00"]`.
2. `best_hour` = phần tử **đầu tiên** của `hours` khớp `s[:2]`.
3. `best_minute` = phần tử đầu tiên của `minutes` khớp `s[3:]`.
4. Ghép `best_hour + ":" + best_minute`.

Chỉ `12 + 60 = 72` thử thay vì `720`.

In [10]:
HOURS_DESC = [f"{h:02d}" for h in range(11, -1, -1)]
MINUTES_DESC = [f"{m:02d}" for m in range(59, -1, -1)]


def find_latest_separate(s: str) -> str:
    best_hour = next(h for h in HOURS_DESC if matches(s[:2], h))       # giờ muộn nhất khớp "HH"
    best_minute = next(m for m in MINUTES_DESC if matches(s[3:], m))   # phút muộn nhất khớp "MM"
    return best_hour + ":" + best_minute


for s, expected in examples:
    print(repr(s), find_latest_separate(s), "(kỳ vọng", expected, ")")

'1?:?4' 11:54 (kỳ vọng 11:54 )
'0?:5?' 09:59 (kỳ vọng 09:59 )
'??:??' 11:59 (kỳ vọng 11:59 )


**Độ phức tạp Cách 2:** **Time `O(1)`** (tối đa `72` thử), **Space `O(1)`**. (`next(...)` lấy phần tử đầu tiên thỏa; sẽ báo `StopIteration` nếu không có — theo đề luôn có.)

### Cách 3 — Thử mọi cách điền `?` bằng `product`, lọc hợp lệ, lấy `max`

**Giải thích.** Gọi `k` là số dấu `?`. Với mỗi tổ hợp `k` chữ số, điền vào các vị trí `?` để được một chuỗi thời gian. Kiểm tra hợp lệ (`giờ <= 11`, `phút <= 59`), rồi `max`. Cách này tổng quát và không cần suy luận luật.

In [11]:
def find_latest_product(s: str) -> str:
    positions = [i for i, c in enumerate(s) if c == "?"]
    best = ""
    for digits in product("0123456789", repeat=len(positions)):
        chars = list(s)
        for pos, d in zip(positions, digits):
            chars[pos] = d                                  # điền một chữ số vào mỗi dấu ?
        t = "".join(chars)
        hour, minute = int(t[:2]), int(t[3:])
        if hour <= 11 and minute <= 59:                     # hợp lệ theo định dạng 12 giờ
            best = max(best, t)                             # so sánh chuỗi = so sánh thời gian
    return best


for s, expected in examples:
    print(repr(s), find_latest_product(s), "(kỳ vọng", expected, ")")

'1?:?4' 11:54 (kỳ vọng 11:54 )
'0?:5?' 09:59 (kỳ vọng 09:59 )
'??:??' 11:59 (kỳ vọng 11:59 )


**Độ phức tạp Cách 3:** **Time `O(10^k · n)`** với `k <= 4` dấu `?` (`10^4 = 10 000`) và `n = 5` ⇒ vẫn hằng số nhỏ; **Space `O(n)`**.

### Cách 4 — Tham lam từng chữ số (code nộp)

**Giải thích.** Áp dụng luật ở §6, xử lý theo thứ tự `H1`, `H2`, `M1`, `M2` (hoặc bất kỳ vì phút độc lập). Đây là cách `O(1)` thao tác, không duyệt gì cả.

In [12]:
def find_latest_greedy(s: str) -> str:
    h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
    if h1 == "?":
        h1 = "1" if (h2 == "?" or h2 <= "1") else "0"       # '1' chỉ khi H2 có thể là 0 hoặc 1
    if h2 == "?":
        h2 = "1" if h1 == "1" else "9"                       # H1 = '1' -> tối đa 11; H1 = '0' -> 09
    if m1 == "?":
        m1 = "5"
    if m2 == "?":
        m2 = "9"
    return f"{h1}{h2}:{m1}{m2}"


for s, expected in examples:
    print(repr(s), find_latest_greedy(s), "(kỳ vọng", expected, ")")

'1?:?4' 11:54 (kỳ vọng 11:54 )
'0?:5?' 09:59 (kỳ vọng 09:59 )
'??:??' 11:59 (kỳ vọng 11:59 )


In [13]:
# Code nộp LeetCode cho Cách 4 (đổi tên lớp thành Solution khi nộp)
class SolutionGreedy:
    def findLatestTime(self, s: str) -> str:
        h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
        if h1 == "?":
            h1 = "1" if (h2 == "?" or h2 <= "1") else "0"
        if h2 == "?":
            h2 = "1" if h1 == "1" else "9"
        if m1 == "?":
            m1 = "5"
        if m2 == "?":
            m2 = "9"
        return f"{h1}{h2}:{m1}{m2}"


print(SolutionGreedy().findLatestTime("0?:5?"))      # 09:59

09:59


**Độ phức tạp Cách 4:** **Time `O(1)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace Cách 4 (tham lam) trên `"1?:?4"`, `"??:??"` và `"?5:??"` (cho thấy `H1` phụ thuộc `H2` thế nào).

In [14]:
def trace_greedy(s: str) -> str:
    h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
    print(f"s = {s!r}: H1={h1!r} H2={h2!r} M1={m1!r} M2={m2!r}")
    if h1 == "?":
        old = h1
        h1 = "1" if (h2 == "?" or h2 <= "1") else "0"
        print(f"  H1 = '?' (H2 gốc = {h2!r}) -> {h1!r}  ({'H2 có thể là 0/1 nên chọn 1' if h1 == '1' else 'H2 >= 2 nên H1 chỉ được 0'})")
    if h2 == "?":
        h2 = "1" if h1 == "1" else "9"
        print(f"  H2 = '?' (H1 = {h1!r}) -> {h2!r}  ({'giờ <= 11' if h2 == '1' else 'giờ 0x: lớn nhất 09'})")
    if m1 == "?":
        m1 = "5"
        print("  M1 = '?' -> '5' (phút <= 59)")
    if m2 == "?":
        m2 = "9"
        print("  M2 = '?' -> '9'")
    result = f"{h1}{h2}:{m1}{m2}"
    print("  ->", result)
    return result


assert trace_greedy("1?:?4") == "11:54"
assert trace_greedy("??:??") == "11:59"
assert trace_greedy("?5:??") == "05:59"

s = '1?:?4': H1='1' H2='?' M1='?' M2='4'
  H2 = '?' (H1 = '1') -> '1'  (giờ <= 11)
  M1 = '?' -> '5' (phút <= 59)
  -> 11:54
s = '??:??': H1='?' H2='?' M1='?' M2='?'
  H1 = '?' (H2 gốc = '?') -> '1'  (H2 có thể là 0/1 nên chọn 1)
  H2 = '?' (H1 = '1') -> '1'  (giờ <= 11)
  M1 = '?' -> '5' (phút <= 59)
  M2 = '?' -> '9'
  -> 11:59
s = '?5:??': H1='?' H2='5' M1='?' M2='?'
  H1 = '?' (H2 gốc = '5') -> '0'  (H2 >= 2 nên H1 chỉ được 0)
  M1 = '?' -> '5' (phút <= 59)
  M2 = '?' -> '9'
  -> 05:59


Bảng các tình huống của `H1`, `H2`:

| `H1` | `H2` | kết quả giờ | giải thích |
|---|---|---|---|
| `?` | `?` | `11` | giờ lớn nhất `11` |
| `?` | `0` hoặc `1` | `10` hoặc `11` | `H1 = 1` được vì `10`, `11` hợp lệ |
| `?` | `2..9` | `02..09` | `H1 = 1` sẽ cho `12+` không hợp lệ ⇒ `H1 = 0` |
| `0` | `?` | `09` | giờ `0x` lớn nhất `09` |
| `1` | `?` | `11` | giờ `1x` lớn nhất `11` |

Phút luôn: `M1 = ?` ⇒ `5`, `M2 = ?` ⇒ `9`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (tham lam): `O(1)`, gọn. (Cách 1 cũng qua dễ dàng nếu bạn chưa nắm luật.)

In [15]:
class Solution:
    def findLatestTime(self, s: str) -> str:
        h1, h2, m1, m2 = s[0], s[1], s[3], s[4]       # bốn chữ số của "HH:MM"
        if h1 == "?":
            # H1 = '1' chỉ khi H2 có thể là 0/1 (giờ 10, 11); nếu H2 >= '2' thì buộc H1 = '0'
            h1 = "1" if (h2 == "?" or h2 <= "1") else "0"
        if h2 == "?":
            h2 = "1" if h1 == "1" else "9"            # 1x -> tối đa 11; 0x -> tối đa 09
        if m1 == "?":
            m1 = "5"                                  # phút tối đa 59
        if m2 == "?":
            m2 = "9"
        return f"{h1}{h2}:{m1}{m2}"


sol = Solution()
for s, expected in examples:
    print(repr(s), "->", sol.findLatestTime(s))

'1?:?4' -> 11:54
'0?:5?' -> 09:59
'??:??' -> 11:59


## 10. Vì sao thuật toán đúng

- **Giờ và phút độc lập:** mọi `HH ∈ [00, 11]` kết hợp được với mọi `MM ∈ [00, 59]`, nên thời gian muộn nhất = (giờ muộn nhất khớp) + (phút muộn nhất khớp).
- **Phút:** mọi cặp `M1M2` với `M1 ≤ 5` đều hợp lệ; nên điền `M2` lớn nhất (`9`) và `M1` lớn nhất (`5`) cho dấu `?`. Nếu một chữ số đã cho cố định thì không đổi; chữ số còn lại điền lớn nhất vẫn hợp lệ (vì `M1 ≤ 5` được đảm bảo cho cố định, `M2` tự do).
- **Giờ:** các giờ hợp lệ là `00..09` và `10, 11`. Với `H1 = ?`: có thể chọn `1` chỉ khi tồn tại `H2` hợp lệ trong `{0, 1}` khớp (`H2 = ?` hoặc `H2 ∈ {0, 1}`), vì `1` luôn tốt hơn `0` ở hàng chục; nếu không thì phải `0`. Với `H2 = ?`: nếu `H1 = 1` thì tối đa là `1` (giờ `11`), nếu `H1 = 0` thì tối đa `9`.
- **Thứ tự `H1` rồi `H2`:** `H1` có trọng số lớn hơn nên được tối ưu trước; quyết định `H1` dùng `H2` **gốc** (khả năng), rồi `H2` tối ưu theo `H1` đã chọn. Đảm bảo bởi kiểm tra vét cạn trên mọi mẫu giờ (§6) và test ngẫu nhiên (§12.3).

## 11. Phân tích Time / Space Complexity

Miền đầu vào là hằng số (`720` thời gian), nên mọi cách đều `O(1)`; dưới đây là số bước tương đối.

| Cách | Số bước (tương đối) | Space |
|---|---|---|
| 1. Duyệt `720` thời gian | `~720 × 5` | `O(1)` |
| 2. Giờ và phút riêng | `~72 × 2` | `O(1)` |
| 3. `product` các `?` | `~10^k × 5` (`k <= 4`) | `O(1)` |
| 4. Tham lam từng chữ số | `~4` | `O(1)` |

## 12. Bộ test

Ta gom các cách vào `impls`; Cách 1 là chuẩn vét cạn.

In [16]:
sol = Solution()
impls = {
    "brute": find_latest_bruteforce,
    "separate": find_latest_separate,
    "product": find_latest_product,
    "greedy": find_latest_greedy,
    "leetcode": sol.findLatestTime,
}


def check_all(s, expected, label):
    for key, fn in impls.items():
        actual = fn(s)
        assert actual == expected, (label, key, s, actual, expected)
    print("PASS", label)


for idx, (s, expected) in enumerate(examples, 1):
    check_all(s, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [17]:
check_all("00:00", "00:00", "không có dấu ?")
check_all("11:59", "11:59", "thời gian lớn nhất, không có ?")
check_all("?0:00", "10:00", "chỉ H1 là ?, H2 = 0")
check_all("?1:00", "11:00", "H2 = 1: H1 chọn 1")
check_all("?2:00", "02:00", "H2 = 2: H1 buộc 0")
check_all("?9:??", "09:59", "H2 = 9: H1 buộc 0")
check_all("1?:00", "11:00", "H1 = 1, H2 = ?: tối đa 11")
check_all("0?:00", "09:00", "H1 = 0, H2 = ?: tối đa 09")
check_all("??:00", "11:00", "cả hai chữ số giờ là ?")
check_all("00:??", "00:59", "chỉ phút là ?")
check_all("00:?0", "00:50", "M2 cố định = 0")
check_all("00:5?", "00:59", "M1 cố định = 5")
check_all("00:0?", "00:09", "M1 cố định = 0")
check_all("?5:?5", "05:55", "H2 = 5 buộc H1 = 0")
check_all("1?:?0", "11:50", "phối hợp")

PASS không có dấu ?
PASS thời gian lớn nhất, không có ?
PASS chỉ H1 là ?, H2 = 0
PASS H2 = 1: H1 chọn 1
PASS H2 = 2: H1 buộc 0
PASS H2 = 9: H1 buộc 0
PASS H1 = 1, H2 = ?: tối đa 11
PASS H1 = 0, H2 = ?: tối đa 09
PASS cả hai chữ số giờ là ?
PASS chỉ phút là ?
PASS M2 cố định = 0
PASS M1 cố định = 5
PASS M1 cố định = 0
PASS H2 = 5 buộc H1 = 0
PASS phối hợp


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **coi là định dạng 24 giờ** (`H1 = 2`, `H2 = 3`); (b) **điền `H1 = '1'` khi `H2` lớn** hoặc **`H2 = '9'` khi `H1 = '1'`**.

In [18]:
def wrong_24h(s):
    h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
    if h1 == "?":
        h1 = "2" if (h2 == "?" or h2 <= "3") else "1"          # SAI: định dạng 24 giờ
    if h2 == "?":
        h2 = "3" if h1 == "2" else "9"
    if m1 == "?":
        m1 = "5"
    if m2 == "?":
        m2 = "9"
    return f"{h1}{h2}:{m1}{m2}"


def wrong_always_nine(s):
    h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
    if h1 == "?":
        h1 = "1"                                                # SAI: không xét H2
    if h2 == "?":
        h2 = "9"                                                # SAI: H1 = 1 thì 19 không hợp lệ
    if m1 == "?":
        m1 = "5"
    if m2 == "?":
        m2 = "9"
    return f"{h1}{h2}:{m1}{m2}"


print(wrong_24h("??:??"), "!=", find_latest_greedy("??:??"))
print(wrong_always_nine("?2:??"), "!=", find_latest_greedy("?2:??"))
print(wrong_always_nine("??:??"), "!=", find_latest_greedy("??:??"))
assert wrong_24h("??:??") != find_latest_greedy("??:??")
assert wrong_always_nine("?2:??") != find_latest_greedy("?2:??")
assert wrong_always_nine("??:??") != find_latest_greedy("??:??")

23:59 != 11:59
12:59 != 02:59
19:59 != 11:59


### 12.3. Đối chiếu ngẫu nhiên và vét cạn mọi mẫu giờ-phút

Cách mạnh nhất: **mọi mẫu hợp lệ**. Mỗi chữ số của `"HH:MM"` là một chữ số `0..9` hoặc `?`: số mẫu `11^4 = 14641`, ta kiểm tra **tất cả** mẫu có ít nhất một thời gian hợp lệ, so Cách 4 (tham lam) với Cách 1 (vét cạn). Thêm vào đó các mẫu sinh ngẫu nhiên từ thời gian thật (che bớt chữ số bằng `?`).

In [19]:
all_patterns = 0
for p in product("0123456789?", repeat=4):
    s = f"{p[0]}{p[1]}:{p[2]}{p[3]}"
    ref = find_latest_bruteforce(s)
    if ref == "":
        continue                                            # không có thời gian hợp lệ: ngoài ràng buộc
    all_patterns += 1
    for key, fn in impls.items():
        assert fn(s) == ref, (key, s)
print("PASS mọi mẫu hợp lệ:", all_patterns, "mẫu (trên tổng 14641)")

rng = random.Random(3114)
for trial in range(2000):
    t = f"{rng.randint(0, 11):02d}:{rng.randint(0, 59):02d}"
    s = "".join(c if c == ":" or rng.random() < 0.5 else "?" for c in t)
    assert len(s) == 5 and s[2] == ":"
    assert find_latest_greedy(s) == find_latest_bruteforce(s)
    assert matches(s, find_latest_greedy(s))
print("PASS 2000 mẫu sinh từ thời gian thật")

PASS mọi mẫu hợp lệ: 1925 mẫu (trên tổng 14641)


PASS 2000 mẫu sinh từ thời gian thật


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Coi là định dạng 24 giờ

In [20]:
def sai_24_gio(s):
    h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
    if h1 == "?":
        h1 = "2"
    if h2 == "?":
        h2 = "3" if h1 == "2" else "9"
    if m1 == "?":
        m1 = "5"
    if m2 == "?":
        m2 = "9"
    return f"{h1}{h2}:{m1}{m2}"


print("sai:", sai_24_gio("??:??"), "| đúng:", Solution().findLatestTime("??:??"))

sai: 23:59 | đúng: 11:59


**Vì sao sai:** đề dùng **12 giờ**: giờ chỉ từ `00` đến `11`. `23:59` không hợp lệ. **Cách sửa:** `H1` tối đa `1`, và khi `H1 = 1` thì `H2` tối đa `1`.

### Lỗi 2: Điền `H2 = '9'` mà quên rằng `H1 = '1'` giới hạn giờ ≤ 11

In [21]:
def sai_h2_luon_9(s):
    h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
    if h1 == "?":
        h1 = "1" if (h2 == "?" or h2 <= "1") else "0"
    if h2 == "?":
        h2 = "9"                                       # SAI: H1 = 1 thì 19 không hợp lệ
    if m1 == "?":
        m1 = "5"
    if m2 == "?":
        m2 = "9"
    return f"{h1}{h2}:{m1}{m2}"


print("sai:", sai_h2_luon_9("1?:??"), "| đúng:", Solution().findLatestTime("1?:??"))

sai: 19:59 | đúng: 11:59


**Vì sao sai:** `19:59` vượt `11:59`. **Cách sửa:** `H2 = '1'` khi `H1 = '1'`; `H2 = '9'` khi `H1 = '0'`.

### Lỗi 3: Điền `H1 = '1'` mà không xét `H2`

In [22]:
def sai_h1_luon_1(s):
    h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
    if h1 == "?":
        h1 = "1"                                       # SAI: nếu H2 = 5 thì 15 không hợp lệ
    if h2 == "?":
        h2 = "1" if h1 == "1" else "9"
    if m1 == "?":
        m1 = "5"
    if m2 == "?":
        m2 = "9"
    return f"{h1}{h2}:{m1}{m2}"


print("sai:", sai_h1_luon_1("?5:??"), "| đúng:", Solution().findLatestTime("?5:??"))

sai: 15:59 | đúng: 05:59


**Vì sao sai:** với `?5`, chọn `H1 = 1` cho `15` — không hợp lệ (giờ ≤ 11). Phải chọn `H1 = 0` (giờ `05`). **Cách sửa:** `H1 = '1'` chỉ khi `H2` là `?` hoặc `'0'`/`'1'`.

### Lỗi 4: Thứ tự xử lý sai khi cả `H1` và `H2` là `?`

In [23]:
def sai_thu_tu(s):
    h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
    if h2 == "?":
        h2 = "9"                                       # SAI: xử lý H2 trước khi biết H1
    if h1 == "?":
        h1 = "1" if h2 <= "1" else "0"
    if m1 == "?":
        m1 = "5"
    if m2 == "?":
        m2 = "9"
    return f"{h1}{h2}:{m1}{m2}"


print("sai:", sai_thu_tu("??:??"), "| đúng:", Solution().findLatestTime("??:??"))

sai: 09:59 | đúng: 11:59


**Vì sao sai:** xử lý `H2` (thành `9`) **trước** làm mất thông tin "H2 còn tự do"; rồi `H1` bị ép về `0`, cho `09:59` thay vì `11:59`. **Cách sửa:** quyết định `H1` bằng giá trị **gốc** của `H2` (`?` nghĩa là còn tự do), rồi mới quyết định `H2` theo `H1`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Miền **nhỏ và hữu hạn** (ở đây `720` thời gian) ⇒ **vét cạn** là cách nộp được ngay, và cũng là **chuẩn kiểm tra** cho lời giải tham lam.
- "**Lớn nhất / muộn nhất** sau khi thay `?`" ⇒ tham lam **từ trái sang phải** (chữ số có trọng số lớn trước), nhưng chú ý chữ số sau có thể **ràng buộc** chữ số trước (như `H1` và `H2`).
- Khi có ràng buộc chéo giữa hai chữ số, **liệt kê bảng các tổ hợp** (như §8) rồi mới viết `if`.
- Luôn kiểm chứng luật tham lam bằng **vét cạn mọi mẫu** (ở đây `11^4` mẫu) — nhỏ và đầy đủ.
- Các biến thể của bài (*Latest Time by Replacing Hidden Digits* 1736) dùng **24 giờ**; đọc kỹ định dạng.

## 15. Biến thể của bài

1. **Latest Time by Replacing Hidden Digits (1736):** định dạng 24 giờ (`00:00`–`23:59`).
2. **Thời gian sớm nhất** sau khi thay `?` (thay `max` bằng `min`).
3. **Đếm số thời gian hợp lệ** có thể tạo ra (*Number of Valid Clock Times*, 2437).
4. **Chuỗi có ngày/tháng** thay vì giờ/phút.
5. **Trả về mọi thời gian hợp lệ khớp mẫu**, sắp xếp tăng dần.

In [24]:
def earliest_time(s):
    for h in range(12):
        for m in range(60):
            t = f"{h:02d}:{m:02d}"
            if matches(s, t):
                return t


def count_valid_times(s):
    return sum(matches(s, f"{h:02d}:{m:02d}") for h in range(12) for m in range(60))


def all_valid_times(s):
    return [f"{h:02d}:{m:02d}" for h in range(12) for m in range(60) if matches(s, f"{h:02d}:{m:02d}")]


def latest_24h(s):
    for h in range(23, -1, -1):
        for m in range(59, -1, -1):
            t = f"{h:02d}:{m:02d}"
            if matches(s, t):
                return t


print(earliest_time("1?:?4"), count_valid_times("1?:?4"), latest_24h("??:??"))   # 10:04 12 23:59
print(all_valid_times("0?:5?")[:3], len(all_valid_times("0?:5?")))
assert earliest_time("1?:?4") == "10:04"
assert count_valid_times("1?:?4") == 2 * 6          # giờ 10, 11 và phút 04,14,24,34,44,54
assert latest_24h("??:??") == "23:59"
assert len(all_valid_times("0?:5?")) == 10 * 10

10:04 12 23:59
['00:50', '00:51', '00:52'] 100


## 16. So sánh các cách làm

| Cách | Ý tưởng | Số bước | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt `720` thời gian | từ muộn tới sớm, khớp mẫu | `~3600` | `O(1)` | chuẩn kiểm chứng, nộp được ngay |
| 2. Giờ và phút riêng | tách `HH` và `MM` | `~144` | `O(1)` | khi muốn nhanh mà vẫn đơn giản |
| 3. `product` các `?` | thử mọi cách điền | `<= 10^4 × 5` | `O(1)` | tổng quát, ít suy luận |
| 4. Tham lam từng chữ số | luật cho `H1`, `H2`, `M1`, `M2` | `~4` | `O(1)` | **nên nộp**, hiểu sâu nhất |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `f"{7:02d}"`, `"3" <= "1"`, `matches("?5", "05")`, `Solution().findLatestTime("?0:?0")`, `Solution().findLatestTime("0?:??")`.

**Bài 2 (tính tay).** Cho các `s` sau, tính thời gian muộn nhất: `"?4:5?"`, `"1?:?9"`, `"?1:?0"`.

**Bài 3 (điền TODO).** Hoàn thành `latest_todo(s)` theo Cách 4 (tham lam).

In [25]:
def latest_todo(s):
    h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
    # TODO 1: nếu h1 == "?": chọn "1" nếu (h2 == "?" hoặc h2 <= "1"), ngược lại "0"
    # TODO 2: nếu h2 == "?": chọn "1" nếu h1 == "1", ngược lại "9"
    # TODO 3: nếu m1 == "?": "5";  nếu m2 == "?": "9"
    # TODO 4: ghép lại thành "HH:MM"
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `latest_regex(s)` dùng **biểu thức chính quy**: đổi `?` thành `.` rồi `re.fullmatch`, duyệt thời gian từ `11:59` giảm dần.

**Bài 5 (biến thể / mini challenge).** (a) Viết `latest_24h_greedy(s)` bằng **tham lam** cho định dạng **24 giờ** (`00:00`–`23:59`), đối chiếu với `latest_24h` bằng vét cạn mọi mẫu. (b) Viết `earliest_greedy(s)` bằng **tham lam** cho thời gian **sớm nhất** (12 giờ), đối chiếu với `earliest_time`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"07"`, `False`, `True`, `"10:50"`, `"09:59"`. Bài 3: chú ý dùng `h2` gốc khi quyết định `h1`.
- **Mức 2 (vừa):** Bài 2: `"?4:5?"` → `H2 = 4 >= 2` nên `H1 = 0`: `04:59`; `"1?:?9"` → `11:59`; `"?1:?0"` → `H2 = 1` nên `H1 = 1`: `11:50`. Bài 4: `pattern = s.replace("?", ".")`, rồi `re.fullmatch(pattern, t)`.
- **Mức 3 (sâu):** Bài 5(a): 24 giờ: `H1 = '?'` ⇒ `'2'` nếu `H2 == '?'` hoặc `H2 <= '3'`, ngược lại `'1'`; `H2 = '?'` ⇒ `'3'` nếu `H1 == '2'`, ngược lại `'9'`; phút như cũ. 5(b): sớm nhất: mọi `?` thành `0`, **trừ** khi sẽ làm giờ không hợp lệ: ở 12 giờ, `0` luôn hợp lệ cho mọi vị trí (00:00 là nhỏ nhất), nên đáp án là thay mọi `?` bằng `0`... nhưng chú ý đề cam kết có thời gian hợp lệ.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
assert f"{7:02d}" == "07"
assert ("3" <= "1") is False
assert matches("?5", "05") is True
assert Solution().findLatestTime("?0:?0") == "10:50"
assert Solution().findLatestTime("0?:??") == "09:59"

# Bài 2
assert Solution().findLatestTime("?4:5?") == "04:59"
assert Solution().findLatestTime("1?:?9") == "11:59"
assert Solution().findLatestTime("?1:?0") == "11:50"


# Bài 3
def latest_todo_ans(s):
    h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
    if h1 == "?":
        h1 = "1" if (h2 == "?" or h2 <= "1") else "0"
    if h2 == "?":
        h2 = "1" if h1 == "1" else "9"
    if m1 == "?":
        m1 = "5"
    if m2 == "?":
        m2 = "9"
    return f"{h1}{h2}:{m1}{m2}"


for s, expected in examples:
    assert latest_todo_ans(s) == expected


# Bài 4
def latest_regex(s):
    pattern = s.replace("?", ".")
    for h in range(11, -1, -1):
        for m in range(59, -1, -1):
            t = f"{h:02d}:{m:02d}"
            if re.fullmatch(pattern, t):
                return t


for s, expected in examples:
    assert latest_regex(s) == expected


# Bài 5(a): 24 giờ
def latest_24h_greedy(s):
    h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
    if h1 == "?":
        h1 = "2" if (h2 == "?" or h2 <= "3") else "1"
    if h2 == "?":
        h2 = "3" if h1 == "2" else "9"
    if m1 == "?":
        m1 = "5"
    if m2 == "?":
        m2 = "9"
    return f"{h1}{h2}:{m1}{m2}"


count_checked = 0
for p in product("0123456789?", repeat=4):
    s = f"{p[0]}{p[1]}:{p[2]}{p[3]}"
    ref = latest_24h(s)
    if ref is None:
        continue
    count_checked += 1
    assert latest_24h_greedy(s) == ref, s
assert count_checked > 1000


# Bài 5(b): sớm nhất (12 giờ)
def earliest_greedy(s):
    return s.replace("?", "0")


for p in product("0123456789?", repeat=4):
    s = f"{p[0]}{p[1]}:{p[2]}{p[3]}"
    ref = earliest_time(s)
    if ref is None:
        continue
    cand = earliest_greedy(s)
    assert cand == ref, (s, cand, ref)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (thay `?` để thời gian 12 giờ muộn nhất)

```python
class Solution:
    def findLatestTime(self, s: str) -> str:
        h1, h2, m1, m2 = s[0], s[1], s[3], s[4]
        if h1 == "?":
            h1 = "1" if (h2 == "?" or h2 <= "1") else "0"
        if h2 == "?":
            h2 = "1" if h1 == "1" else "9"
        if m1 == "?":
            m1 = "5"
        if m2 == "?":
            m2 = "9"
        return f"{h1}{h2}:{m1}{m2}"
```

**Checklist**

- [ ] Giờ tối đa `11` (12 giờ), phút tối đa `59`?
- [ ] `H1 = '?'`: chọn `'1'` chỉ khi `H2` là `'?'` hoặc `<= '1'`, ngược lại `'0'`?
- [ ] `H2 = '?'`: chọn `'1'` nếu `H1 == '1'`, ngược lại `'9'` (quyết định `H1` **trước**, dựa trên `H2` gốc)?
- [ ] `M1 = '?'` ⇒ `'5'`, `M2 = '?'` ⇒ `'9'`?
- [ ] Đã thử `"??:??"`, `"?2:??"`, `"?5:??"`, `"1?:??"`?
- [ ] Skeleton đúng `class Solution` / `def findLatestTime(self, s: str) -> str`?